In [1]:
import os
os.environ['WANDB_DISABLED'] = 'true'    # Kaggle ships wandb and Ultralytics would otherwise ask for a login
os.environ['WANDB_MODE'] = 'disabled'

DATA  = '/kaggle/temp/grocery'           # scratch disk: not saved, doesn't count toward the 20 GB
SHELF = f'{DATA}/ShelfImages'
project_dir = '/kaggle/working'          # saved as notebook output (weights, plots)
os.makedirs(DATA, exist_ok=True)

%pip install -q -U ultralytics
!nvidia-smi -L
print('CPU cores:', os.cpu_count())

Note: you may need to restart the kernel to use updated packages.
GPU 0: Tesla T4 (UUID: GPU-c1942277-d10a-49a3-37e4-a58bec8d0a4c)
GPU 1: Tesla T4 (UUID: GPU-c6654c86-ae74-2d57-d54f-a13672bff71c)
CPU cores: 4


In [2]:
import os, glob, shutil

URL = 'https://github.com/gulvarol/grocerydataset/releases/download/1.0/GroceryDataset_part{}.tar.gz'

if os.path.isdir(SHELF):
    print('images already present, skipping download')
else:
    for i in (1, 2):
        !wget -q -c --show-progress {URL.format(i)} -O {DATA}/part{i}.tar.gz
        !tar -xzf {DATA}/part{i}.tar.gz -C {DATA}
        !rm {DATA}/part{i}.tar.gz

if not os.path.exists(f'{DATA}/annotation.txt'):
    !git clone -q https://github.com/gulvarol/grocerydataset.git /kaggle/temp/grocerydataset_repo
    for fname in ('annotation.txt', 'subset.txt'):
        hits = glob.glob(f'/kaggle/temp/grocerydataset_repo/**/{fname}', recursive=True)
        if hits:
            shutil.copy(hits[0], DATA)
        else:
            print('NOT FOUND in repo:', fname)

print(sorted(os.listdir(DATA)))
print('ShelfImages:', len(os.listdir(SHELF)))      # expect 354

/kaggle/temp/grocer 100%[===================>]   1.41G   240MB/s    in 6.0s    
/kaggle/temp/grocer 100%[===================>]   1.43G   266MB/s    in 5.2s    
['BrandImages', 'BrandImagesFromShelves', 'ProductImages', 'ProductImagesFromShelves', 'ShelfImages', 'annotation.txt', 'subset.txt']
ShelfImages: 355


In [3]:
%pip install -q piexif
import os, re, random, shutil, piexif
from collections import Counter
from PIL import Image

OUT, VAL_FRACTION, SEED = '/kaggle/temp/grocery_yolo', 0.2, 0

# annotation.txt -> {image name: [[x, y, w, h, brand], ...]}
ann = {}
for line in open(f'{DATA}/annotation.txt'):
    t = line.split()
    if t:
        n = int(t[1]); nums = list(map(float, t[2:2 + 5 * n]))
        ann[t[0]] = [nums[i:i + 5] for i in range(0, 5 * n, 5)]

def find_image(name):
    for c in (name, name + '.JPG', name + '.jpg'):
        p = os.path.join(SHELF, c)
        if os.path.exists(p):
            return p

def planogram(name):
    m = re.search(r'_P(\d+)_', name)
    return m.group(1) if m else name

# split by planogram so near-duplicate shots never straddle train and val
planos = sorted({planogram(k) for k in ann})
random.Random(SEED).shuffle(planos)
val_planos = set(planos[:max(1, int(len(planos) * VAL_FRACTION))])

brand_ids = sorted({int(b[4]) for v in ann.values() for b in v})
names = [f'brand_{b}' for b in brand_ids]       # class index = position in brand_ids

shutil.rmtree(OUT, ignore_errors=True)
for s in ('train', 'val'):
    os.makedirs(f'{OUT}/images/{s}'); os.makedirs(f'{OUT}/labels/{s}')

counts = Counter()
for name, boxes in ann.items():
    src = find_image(name)
    assert src, f'missing image: {name}'
    split = 'val' if planogram(name) in val_planos else 'train'
    W, H = Image.open(src).size                  # raw size: what the annotations use
    lines = []
    for x, y, w, h, b in boxes:
        x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
        if x2 - x1 >= 2 and y2 - y1 >= 2:
            lines.append(f'{brand_ids.index(int(b))} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} '
                         f'{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
    dst = f'{OUT}/images/{split}/{os.path.basename(src)}'
    shutil.copy2(src, dst)
    piexif.remove(dst)                           # lossless EXIF strip: trainer sees the raw pixels
    stem = os.path.splitext(os.path.basename(src))[0]
    open(f'{OUT}/labels/{split}/{stem}.txt', 'w').write('\n'.join(lines))
    counts[split] += 1

print(counts, '| classes:', len(names))

Note: you may need to restart the kernel to use updated packages.
Counter({'train': 304, 'val': 50}) | classes: 11


In [4]:
import os, yaml
from ultralytics import YOLO

MODEL = 'yolo26s'

data_yaml = f'{OUT}/data.yaml'
with open(data_yaml, 'w') as f:
    yaml.safe_dump({'path': OUT, 'train': 'images/train', 'val': 'images/val',
                    'names': {i: n for i, n in enumerate(names)}}, f)

model = YOLO(f'{MODEL}.pt')
model.train(
    data=data_yaml,
    imgsz=1280, time=5, patience=30,
    batch=16,
    device=[0, 1],
    max_det=1000,
    workers=min(os.cpu_count(), 4),
    cache='disk',
    project=f'{project_dir}/runs',
    name=f'grocery_{MODEL}',
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/temp/grocery_yolo/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     2/1009      11.2G      1.018      2.298   0.003941        539       1280: 100% ━━━━━━━━━━━━ 19/19 1.8it/s 10.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0it/s 1.0s
                   all         50       1730      0.638      0.255      0.165      0.119

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     3/1164        11G      1.029       1.76   0.003956        485       1280: 100% ━━━━━━━━━━━━ 19/19 1.7it/s 11.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.1it/s 0.9s
                   all         50       1730       0.78      0.256      0.277      0.194

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     4/1223      11.2G      1.022      1.409   0.003828        443       1280: 100% ━━━━━━━━━━━━ 19/1

{'metrics/precision(B)': 0.96191,
 'metrics/recall(B)': 0.96098,
 'metrics/mAP50(B)': 0.97292,
 'metrics/mAP50-95(B)': 0.72454,
 'val/box_loss': 0.96086,
 'val/cls_loss': 0.41182,
 'val/l1_loss': 0.00491,
 'fitness': 0.72454}